# 09 · Registre des modèles (MLflow Model Registry)

In [2]:
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from mlflow import MlflowClient

from evrel.data.load import get_data_dir, load_config
from evrel.models.evaluate import evaluate_ranking
from evrel.models.registry import (MODEL_NAME, RuleScorer, SklearnScorer, as_model_input,
                                   find_run, load_scorer, register)
from evrel.models.train import feature_columns, setup_mlflow

setup_mlflow()
client = MlflowClient()
spec = load_config()["models"]["challenger"]
feats = pd.read_parquet(get_data_dir() / "processed" / "features.parquet")
test = feats[feats["split"] == "test"].copy()
example = as_model_input(test.head(5))
print(f"Registre : {mlflow.get_tracking_uri()}")

Registre : sqlite:///C:/dev/ev-reliability-data/mlflow/mlflow.db


## 1. Enregistrer la règle (repli) et le boosting (champion)

Le boosting enregistré est **exactement** celui du notebook 08 : réentraîné sur l'entraînement et la validation, évalué une fois sur le test. On le récupère depuis son run, on ne le réentraîne pas.

In [3]:
test_run = find_run("test_final", "hgb")
pipeline = mlflow.sklearn.load_model(f"runs:/{test_run}/model")

v_fallback = register(
    RuleScorer(), alias="fallback", input_example=example,
    description="Règle du bon sens : pannes matérielles sur 7 jours. Précision@10 test 17,2 %. Solution de repli.")
v_champion = register(
    SklearnScorer(pipeline, feature_columns(spec["use_site"])), alias="champion", input_example=example,
    source_run_id=test_run,
    description=("Boosting prudent avec le site, entraîné sur janv. 2020 à avr. 2021. "
                 "Précision@10 test 23,5 % (+6,3 pts sur la règle, IC 95 % [+4,1 ; +8,4]). "
                 "Promu champion le 12/10/2026 (notebook 08)."))
print(f"Repli : version {v_fallback}   Champion : version {v_champion}")

2026/10/12 00:26:05 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
2026/10/12 00:26:05 INFO mlflow.pyfunc: Inferring model signature from input example
Successfully registered model 'chargetriage'.
Created version '1' of model 'chargetriage'.
2026/10/12 00:26:10 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
2026/10/12 00:26:10 INFO mlflow.pyfunc: Inferring model signature 

Repli : version 1   Champion : version 2


Registered model 'chargetriage' already exists. Creating a new version of this model...
Created version '2' of model 'chargetriage'.


## 2. Ce que contient le registre

In [4]:
versions = client.search_model_versions(f"name = '{MODEL_NAME}'")
aliases = client.get_registered_model(MODEL_NAME).aliases          # {alias: version}
pd.DataFrame([{
    "version": v.version,
    "alias": ", ".join(a for a, ver in aliases.items() if str(ver) == str(v.version)),
    "créée le (UTC)": pd.to_datetime(v.creation_timestamp, unit="ms").strftime("%d/%m/%Y %H:%M"),
    "description": v.description,
} for v in versions]).sort_values("version")

,version,alias,créée le (UTC),description
1,1,fallback,11/10/2026 22:26,Règle du bon sens : pannes matérielles sur 7 j...
0,2,champion,11/10/2026 22:26,"Boosting prudent avec le site, entraîné sur ja..."


## 3. Vérification : le modèle chargé par son alias est-il bien le bon ?

On compare les scores donnés par « le champion » chargé depuis le registre à ceux du pipeline d'origine, puis on recalcule la précision@10 du test. **Ce n'est pas une nouvelle évaluation** : aucune décision n'en dépend. C'est un contrôle que l'emballage n'a rien changé.

In [5]:
champion = load_scorer("champion")
fallback = load_scorer("fallback")
X = as_model_input(test)

direct = pipeline.predict_proba(X[feature_columns(spec["use_site"])])[:, 1]
via_registry = champion.predict(X)["score"].to_numpy()
print(f"Écart maximal entre pipeline et registre : {np.abs(direct - via_registry).max():.2e}")

check = test.assign(champion=via_registry, fallback=fallback.predict(X)["score"].to_numpy())
pd.DataFrame({
    "Champion (registre)": evaluate_ranking(check, "champion", (10,)),
    "Repli (registre)": evaluate_ranking(check, "fallback", (10,)),
}).loc[["précision@10", "rappel@10"]].round(3)

Écart maximal entre pipeline et registre : 0.00e+00


,Champion (registre),Repli (registre)
précision@10,0.235,0.172
rappel@10,0.353,0.259



Constat : l’écart maximal entre les scores du pipeline d’origine et ceux du champion chargé depuis le registre est de 0,00. Les précisions@10 recalculées (23,5 % pour le champion, 17,2 % pour le repli) sont celles du notebook 08, comme les rappels@10 (35,3 % et 25,9 %).

## 4. Exercice de *rollback* : revenir à la règle en une ligne

Scénario : un soir, le monitoring signale que le modèle se dégrade. On remet l'alias `champion` sur la règle. L'application, qui charge « le champion », utilise alors la règle dès le soir suivant, sans changer une ligne de code. Puis on annule l'exercice.

In [6]:
client.set_registered_model_alias(MODEL_NAME, "champion", v_fallback)
print("Après rollback, le champion donne :", load_scorer("champion").predict(X.head(3))["score"].tolist())

client.set_registered_model_alias(MODEL_NAME, "champion", v_champion)
print("Après retour, le champion donne   :", load_scorer("champion").predict(X.head(3))["score"].round(3).tolist())

Après rollback, le champion donne : [0.0, 0.0, 0.0]
Après retour, le champion donne   : [0.027, 0.024, 0.024]


**Constat** : après le déplacement de l’alias sur la règle, le « champion » renvoie des nombres de pannes (0, 0, 0 : ces trois bornes n’ont eu aucune panne sur 7 jours). Une fois l’alias remis, il renvoie de nouveau des probabilités (2,7 %, 2,4 %, 2,4 %). Aucune ligne de code n’a changé entre les deux.